<a href="https://colab.research.google.com/github/Abdula3469/Laba/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%94%D0%97_%D0%90%D1%82%D0%B0%D0%B5%D0%B2%D0%BE%D0%B9_%D0%B4%D0%BE_7_%D0%B2%D0%BA%D0%BB%D1%8E%D1%87%D0%B8%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D0%BE_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1: Подготовка окружения, импортов и воспроизводимости

# Установка необходимых библиотек
!pip install -q transformers datasets evaluate accelerate torch

import os
import random
import numpy as np
import torch
from transformers import set_seed

def seed_everything(seed=42):
    """Фиксация seed для обеспечения воспроизводимости экспериментов."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)
    print(f"Seed зафиксирован: {seed}")

# Фиксация случайных чисел
seed_everything(42)

# Определение устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# В данной ячейке была настроена среда для выполнения последующих этапов работы.
# Установлены зависимости, импортированы базовые библиотеки и зафиксирован seed для устранения случайности при инициализации весов моделей.
# Также было определено вычислительное устройство, что критически важно для эффективного обучения трансформеров."}],

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.3 MB/s eta 0:00:00
Seed зафиксирован: 42
Используемое устройство: cpu


In [ ]:
from datasets import load_dataset
import pandas as pd

# Загрузка датасета
dataset = load_dataset('rotten_tomatoes')

# Получение train и test сплитов
train_ds = dataset['train']
test_ds = dataset['test']

# Вывод размеров сплитов
print(f"Размер обучающей выборки (train): {len(train_ds)}")
print(f"Размер тестовой выборки (test): {len(test_ds)}")

# Показ примеров записей
print("\nПримеры записей:")
for i in range(2):
    print(f"Пример {i+1}: {train_ds[i]}")

# Подсчет распределения классов
def get_label_distribution(ds):
    # Исправленный способ создания DataFrame из колонки датасета
    df = pd.DataFrame({'label': ds['label']})
    return df['label'].value_counts().sort_index()

train_dist = get_label_distribution(train_ds)
test_dist = get_label_distribution(test_ds)

print("\nРаспределение классов в train:")
print(train_dist)
print("\nРаспределение классов в test:")
print(test_dist)

# В данном задании был загружен датасет Rotten Tomatoes, содержащий кинокритики.
# Мы определили размеры выборок и убедились, что данные сбалансированы (равное количество позитивных и негативных отзывов).
# Задача заключается в бинарной классификации тональности текста: метка 0 соответствует негативному отзыву, а метка 1 — позитивному."}],

Размер обучающей выборки (train): 8530
Размер тестовой выборки (test): 1066

Примеры записей:
Пример 1: {'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}
Пример 2: {'text': 'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .', 'label': 1}

Распределение классов в train:
label
0    4265
1    4265
Name: count, dtype: int64

Распределение классов в test:
label
0    533
1    533
Name: count, dtype: int64


In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

#Загрузка токенизатора
model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

#Функция предобработки
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

# Применение токенизации к датасету
tokenized_datasets = dataset.map(preprocess_function, batched=True)

#Создание Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Просмотр структуры токенизированной записи
print("Структура токенизированной записи:")
sample_entry = tokenized_datasets["train"][0]
for key in sample_entry.keys():
    if key != 'text':
        val = sample_entry[key]
        # Безопасный вывод среза, если это список
        preview = val[:10] if isinstance(val, list) else val
        print(f"{key}: {preview}... (длина/значение: {len(val) if isinstance(val, list) else val})")

# Truncation необходим для того, чтобы тексты, превышающие максимальную длину BERT (512 токенов), обрезались и не вызывали ошибок памяти.
# Динамический padding (через DataCollator) оптимизирует вычисления, дополняя предложения нулями только до длины самого длинного примера в конкретном батче, а не во всем датасете.
# Это значительно ускоряет обучение и снижает расход видеопамяти."}],

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Структура токенизированной записи:
label: 1... (длина/значение: 1)
input_ids: [101, 1103, 2067, 1110, 17348, 1106, 1129, 1103, 6880, 1432]... (длина/значение: 57)
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]... (длина/значение: 57)
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]... (длина/значение: 57)


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

# Загрузка модели
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)
model.to(device)

# Настройка метрик
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels)["f1"]
    return {"accuracy": acc, "f1": f1}

# TrainingArguments
training_args = TrainingArguments(
    output_dir="bert-fine-tuned-rt",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Обучение и оценка
trainer.train()
evaluation_results = trainer.evaluate()

print("\n" + "="*30)
print(f"Итоговые метрики на тесте: {evaluation_results}")
print("="*30)

# В ходе эксперимента был выполнен fine-tuning BERT.
# Высокие значения Accuracy и F1 показывают, что модель успешно адаптировалась к специфике отзывов.
# Возможные ошибки могут быть связаны с короткими саркастичными комментариями."

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.327959,0.366226,0.841463,0.840415


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Итоговые метрики на тесте: {'eval_loss': 0.3662259578704834, 'eval_accuracy': 0.8414634146341463, 'eval_f1': 0.8404154863078376, 'eval_runtime': 160.2447, 'eval_samples_per_second': 6.652, 'eval_steps_per_second': 0.418, 'epoch': 1.0}


In [ ]:
import copy
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Загрузка новой копии модели для эксперимента
model_frozen = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)
model_frozen.to(device)

def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

params_before = count_trainable_parameters(model_frozen)

# Заморозка всех параметров, кроме головы (classifier)
for name, param in model_frozen.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

params_after = count_trainable_parameters(model_frozen)

print(f"Обучаемых параметров до заморозки: {params_before:,}")
print(f"Обучаемых параметров после заморозки: {params_after:,}")

# Настройка Trainer для замороженной модели
training_args_frozen = TrainingArguments(
    output_dir="bert-frozen-rt",
    learning_rate=2e-4, # Увеличим LR, так как обучаем только голову
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none",
    seed=42
)

trainer_frozen = Trainer(
    model=model_frozen,
    args=training_args_frozen,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Обучение
trainer_frozen.train()
frozen_eval_results = trainer_frozen.evaluate()

# Вывод сравнения
print("\nСравнение результатов:")
comparison_data = {
    "Metric": ["Trainable Params", "Accuracy", "F1"],
    "Baseline (Task 4)": [f"{params_before:,}", round(evaluation_results['eval_accuracy'], 4), round(evaluation_results['eval_f1'], 4)],
    "Frozen BERT (Task 5)": [f"{params_after:,}", round(frozen_eval_results['eval_accuracy'], 4), round(frozen_eval_results['eval_f1'], 4)]
}
df_compare = pd.DataFrame(comparison_data)
print(df_compare.to_string(index=False))

# Заморозка BERT значительно сократила количество обучаемых параметров (с ~108 млн до ~1.5 тыс.).
# Это привело к снижению качества по сравнению с полным fine-tuning, так как внутренние веса BERT не адаптировались под специфику текстов отзывов.
# Компромисс: метод Feature Extraction намного быстрее и менее требователен к ресурсам, но почти всегда проигрывает в точности на сложных задачах."}],

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Обучаемых параметров до заморозки: 108,311,810
Обучаемых параметров после заморозки: 1,538


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.683822,0.651529,0.616323,0.648323


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Сравнение результатов:
          Metric Baseline (Task 4) Frozen BERT (Task 5)
Trainable Params       108,311,810                1,538
        Accuracy            0.8415               0.6163
              F1            0.8404               0.6483


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Загрузка новой копии модели
model_partial = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)
model_partial.to(device)

# Заморозка Embeddings и первых 5 блоков энкодера (layer.0 - layer.4)
# В BERT 12 слоев: layer.0 ... layer.11
for name, param in model_partial.named_parameters():
    # Замораживаем эмбеддинги
    if "embeddings" in name:
        param.requires_grad = False
    # Замораживаем первые 5 слоев
    for i in range(5):
        if f"encoder.layer.{i}." in name:
            param.requires_grad = False

params_partial = count_trainable_parameters(model_partial)
print(f"Обучаемых параметров (частичная заморозка 1-5): {params_partial:,}")

# Настройка Trainer
training_args_partial = TrainingArguments(
    output_dir="bert-partial-rt",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none",
    seed=42
)

trainer_partial = Trainer(
    model=model_partial,
    args=training_args_partial,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Обучение
trainer_partial.train()
partial_eval_results = trainer_partial.evaluate()

# Итоговое сравнение всех трех экспериментов
print("\nСравнение всех режимов обучения:")
final_comparison = {
    "Experiment": ["Full Fine-Tuning (T4)", "Frozen Body (T5)", "Partial Freeze 1-5 (T6)"],
    "Trainable Params": [f"{params_before:,}", f"{params_after:,}", f"{params_partial:,}"],
    "Accuracy": [
        round(evaluation_results['eval_accuracy'], 4),
        round(frozen_eval_results['eval_accuracy'], 4),
        round(partial_eval_results['eval_accuracy'], 4)
    ],
    "F1-Score": [
        round(evaluation_results['eval_f1'], 4),
        round(frozen_eval_results['eval_f1'], 4),
        round(partial_eval_results['eval_f1'], 4)
    ]
}
df_final = pd.DataFrame(final_comparison)
display(df_final)

# Частичная заморозка (первых 5 слоев) позволяет сохранить высокую точность, близкую к полному fine-tuning, при этом обновляя на ~40% меньше параметров.
# Это подтверждает гипотезу о том, что нижние слои BERT извлекают универсальные признаки, которые не требуют сильной подстройки,
# в то время как верхние слои критически важны для адаптации к конкретному домену (отзывы о фильмах)."

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Обучаемых параметров (частичная заморозка 1-5): 50,207,234


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.349155,0.369251,0.848030,0.849162


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Сравнение всех режимов обучения:


,Experiment,Trainable Params,Accuracy,F1-Score
0,Full Fine-Tuning (T4),"108,311,810",0.8415,0.8404
1,Frozen Body (T5),"1,538",0.6163,0.6483
2,Partial Freeze 1-5 (T6),"50,207,234",0.8480,0.8492


In [ ]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import time

def freeze_bert_layers(model, n_freeze):
    """
    Замораживает эмбеддинги и первые n_freeze слоев энкодера BERT.
    Если n_freeze = 12, замораживается весь корпус модели (кроме головы).
    """
    # Всегда замораживаем эмбеддинги при n_freeze > 0 для чистоты эксперимента
    if n_freeze > 0:
        for param in model.bert.embeddings.parameters():
            param.requires_grad = False

    # Замораживаем указанное количество слоев
    for i in range(n_freeze):
        for param in model.bert.encoder.layer[i].parameters():
            param.requires_grad = False

    return model

def run_sweep_experiment(n_layers_list):
    sweep_results = []

    for n in n_layers_list:
        print(f"\n>>> Запуск эксперимента: заморозка {n} слоев")

        # Инициализация модели
        model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2).to(device)
        model = freeze_bert_layers(model, n)

        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        args = TrainingArguments(
            output_dir=f"bert-sweep-{n}",
            learning_rate=2e-5 if n < 9 else 1e-4, # Повышаем LR для сильно замороженных моделей
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=1,
            logging_steps=100,
            weight_decay=0.01,
            eval_strategy="no",
            save_strategy="no",
            report_to="none",
            seed=42
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=tokenized_datasets["train"],
            eval_dataset=tokenized_datasets["test"],
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )

        start_time = time.time()
        trainer.train()
        duration = time.time() - start_time

        metrics = trainer.evaluate()

        sweep_results.append({
            "n_freeze": n,
            "Trainable Params": f"{trainable_params:,}",
            "Accuracy": round(metrics["eval_accuracy"], 4),
            "F1": round(metrics["eval_f1"], 4),
            "Time (sec)": round(duration, 2)
        })

    return pd.DataFrame(sweep_results)

# Запуск свипа
layers_to_freeze = [0, 3, 6, 9, 12]
sweep_df = run_sweep_experiment(layers_to_freeze)

# Вывод результатов
print("\nРезультаты мини-исследования:")
display(sweep_df)

# Оптимальной конфигурацией часто является заморозка 3-6 слоев, так как она сохраняет высокую точность при снижении нагрузки на вычисления.
# При полной заморозке (12 слоев) наблюдается резкое падение метрик, так как предобученные веса BERT не всегда идеально подходят под специфику Rotten Tomatoes без адаптации.
# Частичная заморозка позволяет найти баланс (sweet spot) между скоростью сходимости и итоговым качеством классификации.


>>> Запуск эксперимента: заморозка 0 слоев


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.537112
200,0.391488
300,0.377721
400,0.357991
500,0.344386



>>> Запуск эксперимента: заморозка 3 слоев


...

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages

Step,Training Loss
100,0.667814
200,0.601346
300,0.566240
400,0.547002
500,0.537090



Результаты мини-исследования:


,n_freeze,Trainable Params,Accuracy,F1,Time (sec)
0,0,"108,311,810",0.8508,0.8501,4170.02
1,3,"64,382,978",0.8424,0.8418,3378.75
2,6,"43,119,362",0.8386,0.8371,2708.52
3,9,"21,855,746",0.8340,0.8338,2070.20
4,12,"592,130",0.7711,0.7728,1451.77



Интерпретация результатов:
Оптимальной конфигурацией часто является заморозка 3-6 слоев, так как она сохраняет высокую точность при снижении нагрузки на вычисления.
При полной заморозке (12 слоев) наблюдается резкое падение метрик, так как предобученные веса BERT не всегда идеально подходят под специфику Rotten Tomatoes без адаптации.
Частичная заморозка позволяет найти баланс (sweet spot) между скоростью сходимости и итоговым качеством классификации.
